In [0]:
# /// script
# [tool.databricks.environment]
# environment_version = "6"
# ///
# ==============================================================================
# PIPELINE DEPENDENCY & IMPACT TRACKING
# ==============================================================================
# Upstream Dependencies:
#   - [Module]: notebooks/00_config.py        (PATHS, TABLE_NAMES, FOCUS_COUNTRIES, DEFAULT_LOOKBACK_DAYS,
#                                              get_pipeline_parameter, configure_spark_runtime)
#   - [Module]: notebooks/00_audit_logger.py  (PipelineAuditor, layer = "Silver-to-Gold")
#   - [Delta Table]: silver_network_measurements  (notebooks/02_silver_transformation.py)
# Downstream Impact:
#   - [Dimensions]   dim_date_time (hour grain), dim_date (day grain), dim_country_region,
#                    dim_isp_telecom, dim_content_category
#   - [Fact]         fact_network_anomaly            (1 row per measurement showing interference evidence)
#   - [Marts]        agg_daily_censorship_severity, agg_hourly_censorship_severity   (CAI rollups)
#   - [Appends]      pipeline_execution_logs (layer = 'Silver-to-Gold', one row per table)
#   - [Consumers]    Power BI / Databricks SQL dashboards; schema contracts data/schemas/gold_*_schema.json
# Risk & Side Effects:
#   - Refresh is a partition-pruned MERGE per table (idempotent: re-running identical Silver data = 0 changes).
#     The fact and mart tables use WHEN NOT MATCHED BY SOURCE ... DELETE inside the processed date window, so a
#     re-classified measurement leaves fact_network_anomaly and mart counts are recomputed, never double counted.
#   - An EMPTY Silver window never purges Gold (guard) unless allow_empty_purge=True.
#   - Zero-PII: Gold only reads Silver (already PII-free); it holds no IPs, probe ids or raw payloads.
# ==============================================================================

"""
Project Panopticon: Gold Dimensional Modeling (Silver -> Gold)

Star schema (keys follow the Phase 1 proposal ER diagram):

    dim_date_time ─┐
    dim_country_region ─┼──< fact_network_anomaly          (event fact: 1 row per interference observation)
    dim_isp_telecom ─┤
    dim_content_category ─┘
    + agg_daily_censorship_severity / agg_hourly_censorship_severity  (periodic-snapshot marts, CAI)

CAI definitions (all computed from ALL Silver measurements, so the denominator is complete):
    cai_pct                 = 100 * OONI-attested tampered tests / total tests     (headline CAI)
    confirmed_rate_pct      = 100 * CONFIRMED_BLOCKED tests / total tests          (strictest)
    interference_rate_pct   = 100 * tests with ANY evidence (incl. heuristics) / total tests (sensitivity bound)
"""

import importlib
import json
import os
from datetime import date, datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Tuple

try:
    from pyspark.sql import SparkSession, DataFrame, Window
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType, StructField, StringType, LongType, DoubleType,
        IntegerType, BooleanType, TimestampType, DateType,
    )
    from delta.tables import DeltaTable
    HAS_PYSPARK = True
except (ImportError, ModuleNotFoundError):
    SparkSession = DataFrame = DeltaTable = Any  # type: ignore
    HAS_PYSPARK = False

config = None
for _mod_name in ("notebooks.00_config", "00_config"):
    try:
        config = importlib.import_module(_mod_name)
        break
    except (ImportError, ModuleNotFoundError):
        continue
if config is None:
    raise ImportError("[03_gold_dimensional_modeling] Could not import '00_config'.")

audit_logger = None
for _mod_name in ("notebooks.00_audit_logger", "00_audit_logger"):
    try:
        audit_logger = importlib.import_module(_mod_name)
        break
    except (ImportError, ModuleNotFoundError):
        continue

In [0]:
# ==============================================================================
# STEP 0: CONTRACTS (explicit schemas with column comments)
# ==============================================================================

def _f(name: str, dtype: Any, nullable: bool, comment: str) -> "StructField":
    return StructField(name, dtype, nullable, {"comment": comment})


_AUDIT_FIELDS = [
    _f("load_timestamp", TimestampType(), False, "Audit: when the row was inserted / last changed in Gold"),
    _f("_batch_id", StringType(), False, "Audit: Gold batch that last wrote the row"),
]

DIM_DATE_TIME_SCHEMA = StructType([
    _f("date_time_key",   IntegerType(), False, "PK: yyyymmddHH (UTC hour)"),
    _f("date_key",        IntegerType(), False, "FK to dim_date (yyyymmdd)"),
    _f("calendar_date",   DateType(),    False, "UTC calendar date"),
    _f("hour_of_day",     IntegerType(), False, "0-23"),
    _f("day_of_week",     IntegerType(), False, "ISO: 1=Monday .. 7=Sunday"),
    _f("day_name",        StringType(),  False, "Monday .. Sunday"),
    _f("week_of_year",    IntegerType(), False, "ISO week number"),
    _f("month",           IntegerType(), False, "1-12"),
    _f("quarter",         IntegerType(), False, "1-4"),
    _f("year",            IntegerType(), False, "Calendar year"),
    _f("is_weekend",      BooleanType(), False, "True on Saturday/Sunday"),
    _f("is_crisis_period", BooleanType(), False, "True inside a configured CRISIS_PERIODS window"),
    _f("crisis_label",    StringType(),  True,  "Label of the crisis window, if any"),
] + _AUDIT_FIELDS)

DIM_DATE_SCHEMA = StructType([
    _f("date_key",        IntegerType(), False, "PK: yyyymmdd"),
    _f("calendar_date",   DateType(),    False, "UTC calendar date"),
    _f("day_of_week",     IntegerType(), False, "ISO: 1=Monday .. 7=Sunday"),
    _f("day_name",        StringType(),  False, "Monday .. Sunday"),
    _f("week_of_year",    IntegerType(), False, "ISO week number"),
    _f("month",           IntegerType(), False, "1-12"),
    _f("month_name",      StringType(),  False, "January .. December"),
    _f("quarter",         IntegerType(), False, "1-4"),
    _f("year",            IntegerType(), False, "Calendar year"),
    _f("is_weekend",      BooleanType(), False, "True on Saturday/Sunday"),
    _f("is_crisis_period", BooleanType(), False, "True inside a configured CRISIS_PERIODS window"),
    _f("crisis_label",    StringType(),  True,  "Label of the crisis window, if any"),
] + _AUDIT_FIELDS)

DIM_COUNTRY_REGION_SCHEMA = StructType([
    _f("country_code",             StringType(),  False, "PK: ISO-3166 alpha-2 (OONI probe_cc)"),
    _f("country_name",             StringType(),  False, "English short name; falls back to the code"),
    _f("un_region",                StringType(),  False, "UN geoscheme sub-region; 'Unknown' if not curated"),
    _f("press_freedom_index_tier", StringType(),  False, "RSF tier (VERY_SERIOUS...GOOD) or UNKNOWN; see PRESS_FREEDOM_SEED"),
    _f("is_focus_country",         BooleanType(), False, "True for the project's surveillance focus countries"),
] + _AUDIT_FIELDS)

DIM_ISP_TELECOM_SCHEMA = StructType([
    _f("asn_id",         StringType(), False, "PK: 'AS' + autonomous system number; AS0 = unknown"),
    _f("asn_number",     LongType(),   False, "Numeric ASN"),
    _f("provider_name",  StringType(), False, "Most recent probe_network_name; 'Unknown ISP' if none"),
    _f("ownership_type", StringType(), False, "STATE_CONTROLLED | PRIVATE_COMMERCIAL | UNKNOWN (curated seed)"),
    _f("country_code",   StringType(), False, "Country where the ASN is most often observed"),
] + _AUDIT_FIELDS)

DIM_CONTENT_CATEGORY_SCHEMA = StructType([
    _f("category_code",           StringType(), False, "PK: Citizen Lab category code (MISC = unmatched)"),
    _f("category_name",           StringType(), False, "Citizen Lab category description"),
    _f("category_group",          StringType(), False, "Curated theme for dashboards (e.g. CIRCUMVENTION_AND_PRIVACY)"),
    _f("human_rights_risk_tier",  StringType(), False, "HIGH | MEDIUM | LOW (from Silver)"),
] + _AUDIT_FIELDS)

FACT_NETWORK_ANOMALY_SCHEMA = StructType([
    _f("measurement_id",         StringType(),    False, "PK / degenerate dimension: Silver measurement_id"),
    _f("date_time_key",          IntegerType(),   False, "FK to dim_date_time (UTC hour of the measurement)"),
    _f("date_key",               IntegerType(),   False, "FK to dim_date"),
    _f("calendar_date",          DateType(),      False, "UTC date (window / pruning column)"),
    _f("country_code",           StringType(),    False, "FK to dim_country_region"),
    _f("asn_id",                 StringType(),    False, "FK to dim_isp_telecom"),
    _f("category_code",          StringType(),    False, "FK to dim_content_category"),
    _f("test_name",              StringType(),    False, "OONI nettest (degenerate)"),
    _f("test_family",            StringType(),    False, "WEB | MESSAGING | CIRCUMVENTION | OTHER"),
    _f("target_domain",          StringType(),    True,  "Tested host (degenerate)"),
    _f("tampering_vector",       StringType(),    False, "DNS_TAMPERING|TCP_RESET|TLS_DROP|HTTP_BLOCK|ANOMALY_UNCLASSIFIED"),
    _f("classification_status",  StringType(),    False, "CONFIRMED_BLOCKED | HEURISTIC_ANOMALY"),
    _f("dns_anomaly_flag",       IntegerType(),   False, "1 if DNS-layer evidence"),
    _f("tcp_anomaly_flag",       IntegerType(),   False, "1 if TCP-layer evidence"),
    _f("tls_anomaly_flag",       IntegerType(),   False, "1 if TLS-layer evidence"),
    _f("http_anomaly_flag",      IntegerType(),   False, "1 if HTTP-layer evidence"),
    _f("ooni_anomaly_flag",      IntegerType(),   False, "1 if OONI itself flagged an anomaly"),
    _f("ooni_confirmed_flag",    IntegerType(),   False, "1 if OONI confirmed blocking (blockpage fingerprint)"),
    _f("test_runtime_seconds",   DoubleType(),    True,  "Total test harness runtime in seconds (not latency)"),
    _f("taxonomy_version",       StringType(),    False, "Citizen Lab snapshot used for the category"),
    _f("event_timestamp",        TimestampType(), False, "Measurement start (UTC)"),
] + _AUDIT_FIELDS)

_MART_MEASURES = [
    _f("total_measurements",         LongType(),   False, "Tests conducted (CAI denominator)"),
    _f("ooni_attested_tampered",     LongType(),   False, "Tests flagged anomaly or confirmed by OONI (CAI numerator)"),
    _f("confirmed_blocked_count",    LongType(),   False, "Tests with CONFIRMED_BLOCKED status"),
    _f("interference_signal_count",  LongType(),   False, "Tests with ANY evidence (OONI or protocol heuristic)"),
    _f("heuristic_only_count",       LongType(),   False, "Evidence from protocol heuristics only (not OONI-attested)"),
    _f("dns_tampering_count",        LongType(),   False, "Tests whose primary vector is DNS_TAMPERING"),
    _f("tcp_reset_count",            LongType(),   False, "Primary vector TCP_RESET"),
    _f("tls_drop_count",             LongType(),   False, "Primary vector TLS_DROP"),
    _f("http_block_count",           LongType(),   False, "Primary vector HTTP_BLOCK"),
    _f("anomaly_unclassified_count", LongType(),   False, "Anomaly without an identifiable vector"),
    _f("dns_flag_count",             LongType(),   False, "Tests with the DNS flag (flags can co-occur)"),
    _f("tcp_flag_count",             LongType(),   False, "Tests with the TCP flag"),
    _f("tls_flag_count",             LongType(),   False, "Tests with the TLS flag"),
    _f("http_flag_count",            LongType(),   False, "Tests with the HTTP flag"),
    _f("avg_test_runtime_seconds",   DoubleType(), True,  "Mean test_runtime_seconds"),
    _f("cai_pct",                    DoubleType(), False, "Censorship Aggression Index: 100 * attested / total"),
    _f("confirmed_rate_pct",         DoubleType(), False, "100 * confirmed / total"),
    _f("interference_rate_pct",      DoubleType(), False, "100 * any-evidence / total"),
]

AGG_DAILY_SCHEMA = StructType([
    _f("date_key",       IntegerType(), False, "Grain / FK to dim_date"),
    _f("calendar_date",  DateType(),    False, "UTC date"),
    _f("country_code",   StringType(),  False, "Grain / FK to dim_country_region"),
    _f("asn_id",         StringType(),  False, "Grain / FK to dim_isp_telecom"),
    _f("category_code",  StringType(),  False, "Grain / FK to dim_content_category"),
    _f("test_family",    StringType(),  False, "Grain: WEB | MESSAGING | CIRCUMVENTION | OTHER"),
] + _MART_MEASURES + _AUDIT_FIELDS)

AGG_HOURLY_SCHEMA = StructType([
    _f("date_time_key",  IntegerType(), False, "Grain / FK to dim_date_time"),
    _f("calendar_date",  DateType(),    False, "UTC date"),
    _f("hour_of_day",    IntegerType(), False, "0-23"),
    _f("country_code",   StringType(),  False, "Grain / FK to dim_country_region"),
    _f("test_family",    StringType(),  False, "Grain"),
] + _MART_MEASURES + _AUDIT_FIELDS)

GOLD_TABLES: Dict[str, Dict[str, Any]] = {
    "dim_date_time":        {"schema": DIM_DATE_TIME_SCHEMA, "keys": ["date_time_key"], "kind": "dim"},
    "dim_date":             {"schema": DIM_DATE_SCHEMA, "keys": ["date_key"], "kind": "dim"},
    "dim_country_region":   {"schema": DIM_COUNTRY_REGION_SCHEMA, "keys": ["country_code"], "kind": "dim"},
    "dim_isp_telecom":      {"schema": DIM_ISP_TELECOM_SCHEMA, "keys": ["asn_id"], "kind": "dim"},
    "dim_content_category": {"schema": DIM_CONTENT_CATEGORY_SCHEMA, "keys": ["category_code"], "kind": "dim"},
    "fact_network_anomaly": {"schema": FACT_NETWORK_ANOMALY_SCHEMA, "keys": ["measurement_id"], "kind": "fact",
                             "zorder": ["country_code", "date_key", "asn_id"]},
    "agg_daily_censorship_severity": {"schema": AGG_DAILY_SCHEMA, "kind": "mart",
                                      "keys": ["date_key", "country_code", "asn_id", "category_code", "test_family"],
                                      "zorder": ["country_code", "date_key", "asn_id"]},
    "agg_hourly_censorship_severity": {"schema": AGG_HOURLY_SCHEMA, "kind": "mart",
                                       "keys": ["date_time_key", "country_code", "test_family"],
                                       "zorder": ["country_code", "date_time_key"]},
}

SILVER_REQUIRED_COLUMNS: Dict[str, str] = {
    "measurement_id": "string", "event_timestamp": "timestamp", "measurement_date": "date",
    "target_domain": "string", "probe_asn": "bigint", "probe_cc": "string", "probe_network_name": "string",
    "test_name": "string", "tampering_vector": "string", "classification_status": "string",
    "ooni_anomaly": "boolean", "ooni_confirmed": "boolean", "is_anomaly": "boolean",
    "dns_anomaly_flag": "int", "tcp_anomaly_flag": "int", "tls_anomaly_flag": "int", "http_anomaly_flag": "int",
    "test_runtime_seconds": "double", "content_category": "string", "category_description": "string",
    "human_rights_risk_tier": "string", "taxonomy_version": "string", "load_timestamp": "timestamp",
}

# --- Curated reference seeds (documented, extendable) -----------------------------------------
# UN geoscheme sub-regions (objective). Unlisted codes fall back to 'Unknown' and the code as the name.
COUNTRY_SEED: Dict[str, Tuple[str, str]] = {
    "PK": ("Pakistan", "Southern Asia"), "IR": ("Iran", "Southern Asia"), "RU": ("Russia", "Eastern Europe"),
    "IN": ("India", "Southern Asia"), "BD": ("Bangladesh", "Southern Asia"), "AF": ("Afghanistan", "Southern Asia"),
    "CN": ("China", "Eastern Asia"), "VN": ("Vietnam", "South-eastern Asia"), "MM": ("Myanmar", "South-eastern Asia"),
    "TR": ("Turkey", "Western Asia"), "SA": ("Saudi Arabia", "Western Asia"), "AE": ("United Arab Emirates", "Western Asia"),
    "EG": ("Egypt", "Northern Africa"), "BY": ("Belarus", "Eastern Europe"), "UA": ("Ukraine", "Eastern Europe"),
    "KZ": ("Kazakhstan", "Central Asia"), "UZ": ("Uzbekistan", "Central Asia"), "TM": ("Turkmenistan", "Central Asia"),
    "US": ("United States", "Northern America"), "GB": ("United Kingdom", "Northern Europe"),
    "DE": ("Germany", "Western Europe"), "FR": ("France", "Western Europe"), "NL": ("Netherlands", "Western Europe"),
    "ZZ": ("Unknown country", "Unknown"),
}
# RSF World Press Freedom Index 2025 red zone ("very serious"): India, China, Pakistan, Bangladesh, Russia named in
# https://rsf.org/en/world-press-freedom-index-2025-over-half-worlds-population-red-zones ; Iran, Egypt, Vietnam,
# Afghanistan, Turkmenistan listed there with scores below 25/100. Anything not listed stays UNKNOWN (never guessed).
# The 2026 index has since been published (e.g. Russia 172nd, Iran 177th, Pakistan 153rd): refresh this seed from it.
PRESS_FREEDOM_SEED: Dict[str, str] = {
    "PK": "VERY_SERIOUS", "IR": "VERY_SERIOUS", "RU": "VERY_SERIOUS", "IN": "VERY_SERIOUS", "CN": "VERY_SERIOUS",
    "BD": "VERY_SERIOUS", "EG": "VERY_SERIOUS", "VN": "VERY_SERIOUS", "AF": "VERY_SERIOUS", "TM": "VERY_SERIOUS",
}
# Ownership of well-known carriers. Only entries that are widely documented are seeded; extend after verification.
ISP_OWNERSHIP_SEED: Dict[str, str] = {
    "AS17557": "STATE_CONTROLLED",   # Pakistan Telecommunication Company Ltd (PTCL): Government of Pakistan majority
    "AS12389": "STATE_CONTROLLED",   # Rostelecom: Russian state-controlled
}
# Citizen Lab category codes: (name, dashboard theme). Tier is NOT here: it comes from Silver / SILVER_TIER_*.
CATEGORY_SEED: Dict[str, Tuple[str, str]] = {
    "NEWS": ("News Media", "POLITICAL_AND_NEWS"), "POLR": ("Political Criticism", "POLITICAL_AND_NEWS"),
    "MILX": ("Terrorism and Militants", "POLITICAL_AND_NEWS"), "GOVT": ("Government", "POLITICAL_AND_NEWS"),
    "IGO": ("Intergovernmental Organizations", "POLITICAL_AND_NEWS"),
    "HUMR": ("Human Rights Issues", "RIGHTS_AND_ADVOCACY"), "LGBT": ("LGBT", "RIGHTS_AND_ADVOCACY"),
    "ENV": ("Environment", "RIGHTS_AND_ADVOCACY"), "PUBH": ("Public Health", "RIGHTS_AND_ADVOCACY"),
    "XED": ("Sex Education", "RIGHTS_AND_ADVOCACY"),
    "ANON": ("Anonymization and Circumvention Tools", "CIRCUMVENTION_AND_PRIVACY"),
    "HACK": ("Hacking Tools", "CIRCUMVENTION_AND_PRIVACY"),
    "COMT": ("Communication Tools", "COMMUNICATION_AND_SOCIAL"), "GRP": ("Social Networking", "COMMUNICATION_AND_SOCIAL"),
    "MMED": ("Media Sharing", "COMMUNICATION_AND_SOCIAL"), "HOST": ("Hosting and Blogging Platforms", "COMMUNICATION_AND_SOCIAL"),
    "DATE": ("Online Dating", "COMMUNICATION_AND_SOCIAL"), "SRCH": ("Search Engines", "COMMUNICATION_AND_SOCIAL"),
    "REL": ("Religion", "RELIGION_AND_CULTURE"), "CULTR": ("Culture", "RELIGION_AND_CULTURE"),
    "PORN": ("Pornography", "SENSITIVE_CONTENT"), "PROV": ("Provocative Attire", "SENSITIVE_CONTENT"),
    "GMB": ("Gambling", "SENSITIVE_CONTENT"), "ALDR": ("Alcohol and Drugs", "SENSITIVE_CONTENT"),
    "HATE": ("Hate Speech", "SENSITIVE_CONTENT"),
    "COMM": ("E-commerce", "COMMERCE_AND_ENTERTAINMENT"), "ECON": ("Economics", "COMMERCE_AND_ENTERTAINMENT"),
    "GAME": ("Gaming", "COMMERCE_AND_ENTERTAINMENT"), "FILE": ("File-sharing", "COMMERCE_AND_ENTERTAINMENT"),
    "MISC": ("Miscellaneous / Uncategorized", "OTHER"), "CTRL": ("Control Content", "OTHER"),
}
# MUST stay identical to RISK_TIER_HIGH / RISK_TIER_MEDIUM in 02_silver_transformation.py (a test enforces this).
SILVER_TIER_HIGH = ["POLR", "HUMR", "NEWS", "LGBT", "MMED"]
SILVER_TIER_MEDIUM = ["ANON", "COMM", "REL", "FILE", "PUBH"]

TEST_FAMILY_MAP: Dict[str, List[str]] = {
    "WEB": ["web_connectivity"],
    "MESSAGING": ["whatsapp", "telegram", "signal", "facebook_messenger"],
    "CIRCUMVENTION": ["tor", "torsf", "vanilla_tor", "psiphon"],
}
# Editable: (start_date, end_date, label) windows flagged as crisis periods in dim_date / dim_date_time.
CRISIS_PERIODS: List[Tuple[str, str, str]] = []

GOLD_LAYER_NAME = "Silver-to-Gold"

In [0]:
# ==============================================================================
# STEP 1: PARAMETERS, SESSION, PATHS
# ==============================================================================

def init_gold_widgets() -> None:
    """Databricks widgets (same names as Silver so one Job can pass identical parameters to every stage)."""
    try:
        import IPython
        ipy = IPython.get_ipython()
        if ipy and "dbutils" in ipy.user_ns:
            w = ipy.user_ns["dbutils"].widgets
            yesterday = (datetime.now(timezone.utc).date() - timedelta(days=1)).isoformat()
            w.text("batch_date", yesterday, "1. Batch Date T (YYYY-MM-DD)")
            w.dropdown("load_type", "incremental", ["full", "incremental", "backfill"], "2. Load Type")
            w.text("lookback_days", str(getattr(config, "DEFAULT_LOOKBACK_DAYS", 3)), "3. Rolling Lookback Days")
            w.text("backfill_start", "", "4. Backfill Start (YYYY-MM-DD, optional)")
            w.text("backfill_end", "", "5. Backfill End (YYYY-MM-DD, optional)")
            w.text("country_filter", "ALL", "6. Country Filter (ALL or PK/IR/RU)")
    except Exception:
        pass


def _parse_date(value: Any, name: str) -> date:
    try:
        return datetime.strptime(str(value).strip(), "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(f"[Parameter Error] '{name}' must be YYYY-MM-DD, got '{value}'.")


def resolve_gold_parameters(
    batch_date: Optional[str] = None, load_type: Optional[str] = None, lookback_days: Optional[int] = None,
    backfill_start: Optional[str] = None, backfill_end: Optional[str] = None,
    country_filter: Optional[str] = None, window_start: Optional[str] = None, window_end: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Same window semantics as Silver. `window_start/window_end` (passed by the orchestrator from the Silver
    summary) take precedence so Gold refreshes exactly the dates Silver just processed.
    """
    get = config.get_pipeline_parameter
    yesterday = (datetime.now(timezone.utc).date() - timedelta(days=1)).isoformat()
    r_load = str(load_type or get("load_type", "incremental")).lower()
    if r_load not in ("full", "incremental", "backfill"):
        raise ValueError(f"[Parameter Error] load_type must be full|incremental|backfill, got '{r_load}'.")
    r_batch = _parse_date(batch_date or get("batch_date", yesterday), "batch_date")
    r_lookback = int(lookback_days if lookback_days is not None
                     else get("lookback_days", getattr(config, "DEFAULT_LOOKBACK_DAYS", 3)))
    if r_lookback < 0:
        raise ValueError("[Parameter Error] lookback_days must be >= 0.")
    r_country = str(country_filter or get("country_filter", "ALL")).strip().upper() or "ALL"
    r_bf_start = str(backfill_start or get("backfill_start", "") or "").strip()
    r_bf_end = str(backfill_end or get("backfill_end", "") or "").strip()

    if window_start and window_end:
        w_start, w_end = _parse_date(window_start, "window_start"), _parse_date(window_end, "window_end")
    elif r_load == "full":
        w_start = w_end = None
    elif r_load == "backfill" and r_bf_start:
        w_start = _parse_date(r_bf_start, "backfill_start")
        w_end = _parse_date(r_bf_end, "backfill_end") if r_bf_end else w_start
    else:
        w_end = r_batch
        w_start = r_batch - timedelta(days=r_lookback)
    if w_start and w_end and w_start > w_end:
        raise ValueError(f"[Parameter Error] window start {w_start} is after end {w_end}.")

    if w_start is None:
        batch_id = f"GOLD_FULL_{r_country}"
    else:
        batch_id = f"GOLD_{w_start:%Y%m%d}_{w_end:%Y%m%d}_{r_country}_{r_load.upper()}"
    return {"batch_date": r_batch.isoformat(), "load_type": r_load, "lookback_days": r_lookback,
            "window_start": w_start.isoformat() if w_start else None,
            "window_end": w_end.isoformat() if w_end else None,
            "country_filter": r_country, "batch_id": batch_id}


def _safe_conf_set(conf: Any, key: str, value: str) -> None:
    try:
        conf.set(key, value)
    except Exception:
        pass


def resolve_spark_session(spark: Optional[Any] = None) -> Any:
    session = spark or SparkSession.builder.getOrCreate()
    try:
        config.configure_spark_runtime(session)
    except Exception:
        pass
    _safe_conf_set(session.conf, "spark.sql.shuffle.partitions", "4")
    _safe_conf_set(session.conf, "spark.sql.session.timeZone", "UTC")
    _safe_conf_set(session.conf, "spark.sql.parquet.compression.codec", "snappy")
    _safe_conf_set(session.conf, "spark.databricks.delta.optimizeWrite.enabled", "true")
    _safe_conf_set(session.conf, "spark.databricks.delta.autoCompact.enabled", "true")
    return session


def gold_path(table: str) -> str:
    """config.PATHS['gold_<table>'] if defined, else <STORAGE_ROOT>/gold/<table> (sibling of the silver folder)."""
    paths = getattr(config, "PATHS", {}) or {}
    if paths.get(f"gold_{table}"):
        return paths[f"gold_{table}"]
    root = paths["silver"].rstrip("/").rsplit("/", 2)[0]
    return f"{root}/gold/{table}"


def gold_table_name(table: str) -> str:
    return (getattr(config, "TABLE_NAMES", {}) or {}).get(f"gold_{table}", table)

In [0]:
# ==============================================================================
# STEP 2: READ SILVER + CONTRACT GUARD
# ==============================================================================

class SilverContractError(RuntimeError):
    """Raised when Silver lacks a column Gold cannot work without."""


def read_silver_window(spark, silver_path: str, window_start: Optional[str], window_end: Optional[str],
                       country_filter: str) -> "DataFrame":
    df = spark.read.format("delta").load(silver_path)
    if window_start and window_end:
        df = df.filter((F.col("measurement_date") >= F.lit(window_start).cast("date")) &
                       (F.col("measurement_date") <= F.lit(window_end).cast("date")))
    if country_filter != "ALL":
        df = df.filter(F.upper(F.col("probe_cc")) == country_filter)
    return df


def validate_silver_input(df: "DataFrame") -> "DataFrame":
    """Fail fast on a missing column; coerce changed types with try_cast; project to the columns Gold uses."""
    actual = {f.name: f.dataType.simpleString() for f in df.schema.fields}
    missing = sorted(set(SILVER_REQUIRED_COLUMNS) - set(actual))
    if missing:
        raise SilverContractError(f"Silver is missing required columns: {missing}. Re-run the Silver layer first.")
    return df.select(*[
        (F.col(c) if actual[c] == t else F.expr(f"try_cast(`{c}` AS {t})")).alias(c)
        for c, t in SILVER_REQUIRED_COLUMNS.items()
    ])

In [0]:
# ==============================================================================
# STEP 3: PURE TRANSFORMATIONS (no I/O; unit-testable)
# ==============================================================================

def family_of_test(test_col: "F.Column") -> "F.Column":
    expr = F.lit("OTHER")
    for family, names in reversed(list(TEST_FAMILY_MAP.items())):
        expr = F.when(test_col.isin(*names), F.lit(family)).otherwise(expr)
    return expr


def risk_tier_expr(code_col: "F.Column") -> "F.Column":
    return (F.when(code_col.isin(*SILVER_TIER_HIGH), F.lit("HIGH"))
             .when(code_col.isin(*SILVER_TIER_MEDIUM), F.lit("MEDIUM")).otherwise(F.lit("LOW")))


def _stamp(df: "DataFrame", schema: "StructType", batch_id: str, load_ts: datetime) -> "DataFrame":
    df = df.withColumn("load_timestamp", F.lit(load_ts).cast("timestamp")).withColumn("_batch_id", F.lit(batch_id))
    return df.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in schema.fields])


def with_gold_keys(df: "DataFrame") -> "DataFrame":
    """Adds the conformed keys shared by facts and marts (Silver is the only input)."""
    return (df
        .withColumn("calendar_date", F.col("measurement_date"))
        .withColumn("date_key", F.date_format(F.col("measurement_date"), "yyyyMMdd").cast("int"))
        .withColumn("date_time_key", F.date_format(F.col("event_timestamp"), "yyyyMMddHH").cast("int"))
        .withColumn("hour_of_day", F.hour(F.col("event_timestamp")).cast("int"))
        .withColumn("country_code", F.upper(F.trim(F.col("probe_cc"))))
        .withColumn("asn_id", F.concat(F.lit("AS"), F.coalesce(F.col("probe_asn"), F.lit(0)).cast("string")))
        .withColumn("category_code", F.coalesce(F.upper(F.trim(F.col("content_category"))), F.lit("MISC")))
        .withColumn("test_family", family_of_test(F.col("test_name"))))


def _crisis_columns(date_col: "F.Column") -> Tuple["F.Column", "F.Column"]:
    flag, label = F.lit(False), F.lit(None).cast("string")
    for start, end, name in reversed(CRISIS_PERIODS):
        inside = (date_col >= F.lit(start).cast("date")) & (date_col <= F.lit(end).cast("date"))
        flag = F.when(inside, F.lit(True)).otherwise(flag)
        label = F.when(inside, F.lit(name)).otherwise(label)
    return flag, label


def _calendar_columns(df: "DataFrame") -> "DataFrame":
    iso_dow = ((F.dayofweek(F.col("calendar_date")) + 5) % 7 + 1)
    crisis, crisis_label = _crisis_columns(F.col("calendar_date"))
    return (df
        .withColumn("date_key", F.date_format(F.col("calendar_date"), "yyyyMMdd").cast("int"))
        .withColumn("day_of_week", iso_dow.cast("int"))
        .withColumn("day_name", F.date_format(F.col("calendar_date"), "EEEE"))
        .withColumn("week_of_year", F.weekofyear(F.col("calendar_date")).cast("int"))
        .withColumn("month", F.month(F.col("calendar_date")).cast("int"))
        .withColumn("month_name", F.date_format(F.col("calendar_date"), "MMMM"))
        .withColumn("quarter", F.quarter(F.col("calendar_date")).cast("int"))
        .withColumn("year", F.year(F.col("calendar_date")).cast("int"))
        .withColumn("is_weekend", iso_dow >= 6)
        .withColumn("is_crisis_period", crisis).withColumn("crisis_label", crisis_label))


def build_dim_date(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    days = df_silver.select(F.col("measurement_date").alias("calendar_date")).where(F.col("calendar_date").isNotNull()).distinct()
    return _stamp(_calendar_columns(days), DIM_DATE_SCHEMA, batch_id, load_ts)


def build_dim_date_time(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    """Every hour of every date present in the window (also hours without measurements, so blackouts show as 0)."""
    days = df_silver.select(F.col("measurement_date").alias("calendar_date")).where(F.col("calendar_date").isNotNull()).distinct()
    hours = df_silver.sparkSession.range(24).select(F.col("id").cast("int").alias("hour_of_day"))
    grid = _calendar_columns(days.crossJoin(hours)).withColumn(
        "date_time_key", (F.col("date_key") * 100 + F.col("hour_of_day")).cast("int"))
    return _stamp(grid, DIM_DATE_TIME_SCHEMA, batch_id, load_ts)


def _seed_df(spark, rows: List[Tuple], cols: List[str]) -> "DataFrame":
    return spark.createDataFrame(rows, ", ".join(f"{c} string" for c in cols))


def build_dim_country_region(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    spark = df_silver.sparkSession
    seed = _seed_df(spark, [(k, v[0], v[1], PRESS_FREEDOM_SEED.get(k, "UNKNOWN")) for k, v in COUNTRY_SEED.items()],
                    ["code", "seed_name", "seed_region", "seed_tier"])
    focus = list(getattr(config, "FOCUS_COUNTRIES", ["PK", "IR", "RU"]))
    codes = (df_silver.select(F.upper(F.trim(F.col("probe_cc"))).alias("country_code")).where(F.col("country_code").isNotNull()).distinct())
    out = (codes.join(F.broadcast(seed), codes["country_code"] == seed["code"], "left")
        .withColumn("country_name", F.coalesce(F.col("seed_name"), F.col("country_code")))
        .withColumn("un_region", F.coalesce(F.col("seed_region"), F.lit("Unknown")))
        .withColumn("press_freedom_index_tier", F.coalesce(F.col("seed_tier"), F.lit("UNKNOWN")))
        .withColumn("is_focus_country", F.col("country_code").isin(*focus)))
    return _stamp(out, DIM_COUNTRY_REGION_SCHEMA, batch_id, load_ts)


def build_dim_isp_telecom(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    spark = df_silver.sparkSession
    k = with_gold_keys(df_silver).withColumn("asn_number", F.coalesce(F.col("probe_asn"), F.lit(0)).cast("long"))
    w_name = Window.partitionBy("asn_id").orderBy(F.col("event_timestamp").desc_nulls_last())
    names = (k.where(F.col("probe_network_name").isNotNull() & (F.trim(F.col("probe_network_name")) != ""))
              .withColumn("_r", F.row_number().over(w_name)).where(F.col("_r") == 1)
              .select("asn_id", F.col("probe_network_name").alias("provider_name")))
    w_cc = Window.partitionBy("asn_id").orderBy(F.col("_n").desc(), F.col("country_code").asc())
    ccs = (k.groupBy("asn_id", "country_code").agg(F.count("*").alias("_n"))
             .withColumn("_r", F.row_number().over(w_cc)).where(F.col("_r") == 1).select("asn_id", "country_code"))
    base = k.select("asn_id", "asn_number").distinct()
    owners = _seed_df(spark, list(ISP_OWNERSHIP_SEED.items()), ["seed_asn", "seed_owner"])
    out = (base.join(names, "asn_id", "left").join(ccs, "asn_id", "left")
        .join(F.broadcast(owners), base["asn_id"] == owners["seed_asn"], "left")
        .withColumn("provider_name", F.coalesce(F.col("provider_name"), F.lit("Unknown ISP")))
        .withColumn("ownership_type", F.coalesce(F.col("seed_owner"), F.lit("UNKNOWN")))
        .withColumn("country_code", F.coalesce(F.col("country_code"), F.lit("ZZ"))))
    return _stamp(out, DIM_ISP_TELECOM_SCHEMA, batch_id, load_ts)


def build_dim_content_category(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    """All Citizen Lab codes (so the dimension is complete) overlaid with what Silver actually carries."""
    spark = df_silver.sparkSession
    seed = _seed_df(spark, [(c, v[0], v[1]) for c, v in CATEGORY_SEED.items()], ["category_code", "seed_name", "seed_group"])
    w = Window.partitionBy("category_code").orderBy(F.col("load_timestamp").desc_nulls_last())
    observed = (df_silver.select(F.coalesce(F.upper(F.trim(F.col("content_category"))), F.lit("MISC")).alias("category_code"),
                                 F.col("category_description").alias("obs_name"),
                                 F.col("human_rights_risk_tier").alias("obs_tier"), F.col("load_timestamp"))
                .withColumn("_r", F.row_number().over(w)).where(F.col("_r") == 1).drop("_r", "load_timestamp"))
    codes = seed.select("category_code").union(observed.select("category_code")).distinct()
    out = (codes.join(seed, "category_code", "left").join(observed, "category_code", "left")
        .withColumn("category_name", F.coalesce(F.col("obs_name"), F.col("seed_name"), F.col("category_code")))
        .withColumn("category_group", F.coalesce(F.col("seed_group"), F.lit("OTHER")))
        .withColumn("human_rights_risk_tier", F.coalesce(F.col("obs_tier"), risk_tier_expr(F.col("category_code")))))
    return _stamp(out, DIM_CONTENT_CATEGORY_SCHEMA, batch_id, load_ts)


def build_fact_network_anomaly(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    """Event fact: one row per measurement that shows interference evidence (is_anomaly)."""
    k = with_gold_keys(df_silver.where(F.col("is_anomaly")))
    out = (k
        .withColumn("ooni_anomaly_flag", F.col("ooni_anomaly").cast("int"))
        .withColumn("ooni_confirmed_flag", F.col("ooni_confirmed").cast("int")))
    return _stamp(out, FACT_NETWORK_ANOMALY_SCHEMA, batch_id, load_ts)


def aggregate_severity(df_keyed: "DataFrame", group_cols: List[str]) -> "DataFrame":
    """Shared rollup for the daily and hourly marts (denominator = ALL measurements)."""
    one = lambda cond: F.sum(F.when(cond, F.lit(1)).otherwise(F.lit(0))).cast("long")   # noqa: E731
    attested = F.col("ooni_anomaly") | F.col("ooni_confirmed")
    agg = df_keyed.groupBy(*group_cols).agg(
        F.count(F.lit(1)).cast("long").alias("total_measurements"),
        one(attested).alias("ooni_attested_tampered"),
        one(F.col("classification_status") == "CONFIRMED_BLOCKED").alias("confirmed_blocked_count"),
        one(F.col("is_anomaly")).alias("interference_signal_count"),
        one(F.col("is_anomaly") & ~attested).alias("heuristic_only_count"),
        one(F.col("tampering_vector") == "DNS_TAMPERING").alias("dns_tampering_count"),
        one(F.col("tampering_vector") == "TCP_RESET").alias("tcp_reset_count"),
        one(F.col("tampering_vector") == "TLS_DROP").alias("tls_drop_count"),
        one(F.col("tampering_vector") == "HTTP_BLOCK").alias("http_block_count"),
        one(F.col("tampering_vector") == "ANOMALY_UNCLASSIFIED").alias("anomaly_unclassified_count"),
        one(F.col("dns_anomaly_flag") == 1).alias("dns_flag_count"),
        one(F.col("tcp_anomaly_flag") == 1).alias("tcp_flag_count"),
        one(F.col("tls_anomaly_flag") == 1).alias("tls_flag_count"),
        one(F.col("http_anomaly_flag") == 1).alias("http_flag_count"),
        F.avg(F.col("test_runtime_seconds")).alias("avg_test_runtime_seconds"))
    pct = lambda c: F.round(F.col(c) * 100.0 / F.col("total_measurements"), 4)   # noqa: E731
    return (agg.withColumn("cai_pct", pct("ooni_attested_tampered"))
               .withColumn("confirmed_rate_pct", pct("confirmed_blocked_count"))
               .withColumn("interference_rate_pct", pct("interference_signal_count")))


def build_agg_daily(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    k = with_gold_keys(df_silver)
    agg = aggregate_severity(k, ["date_key", "calendar_date", "country_code", "asn_id", "category_code", "test_family"])
    return _stamp(agg, AGG_DAILY_SCHEMA, batch_id, load_ts)


def build_agg_hourly(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> "DataFrame":
    k = with_gold_keys(df_silver)
    agg = aggregate_severity(k, ["date_time_key", "calendar_date", "hour_of_day", "country_code", "test_family"])
    return _stamp(agg, AGG_HOURLY_SCHEMA, batch_id, load_ts)


def transform_silver_to_gold(df_silver: "DataFrame", batch_id: str, load_ts: datetime) -> Dict[str, "DataFrame"]:
    """All eight Gold frames, each conforming to its schema contract."""
    return {
        "dim_date_time": build_dim_date_time(df_silver, batch_id, load_ts),
        "dim_date": build_dim_date(df_silver, batch_id, load_ts),
        "dim_country_region": build_dim_country_region(df_silver, batch_id, load_ts),
        "dim_isp_telecom": build_dim_isp_telecom(df_silver, batch_id, load_ts),
        "dim_content_category": build_dim_content_category(df_silver, batch_id, load_ts),
        "fact_network_anomaly": build_fact_network_anomaly(df_silver, batch_id, load_ts),
        "agg_daily_censorship_severity": build_agg_daily(df_silver, batch_id, load_ts),
        "agg_hourly_censorship_severity": build_agg_hourly(df_silver, batch_id, load_ts),
    }

In [0]:
# ==============================================================================
# STEP 4: SCHEMA CONTRACTS
# ==============================================================================

def schema_to_contract(schema: "StructType", title: str, description: str, keys: List[str]) -> Dict[str, Any]:
    return {"title": title, "description": description, "type": "struct", "primary_key": keys,
            "fields": [{"name": f.name, "type": f.dataType.typeName(), "nullable": f.nullable,
                        "metadata": dict(f.metadata or {})} for f in schema.fields]}


def validate_gold_schema_contract(df: "DataFrame", expected_schema_json_path: str) -> Tuple[bool, List[str]]:
    with open(expected_schema_json_path, "r", encoding="utf-8") as fh:
        contract = json.load(fh)
    expected = {x["name"]: x for x in contract["fields"]}
    actual = {f.name: f for f in df.schema.fields}
    errors: List[str] = []
    for name, spec in expected.items():
        if name not in actual:
            errors.append(f"Missing column: '{name}'")
        elif actual[name].dataType.typeName().lower() != str(spec["type"]).lower():
            errors.append(f"Type mismatch on '{name}': {actual[name].dataType.typeName()} != {spec['type']}")
    errors += [f"Unexpected extra column: '{n}'" for n in actual if n not in expected]
    return (not errors), errors

In [0]:
# ==============================================================================
# STEP 5: DELTA PERSISTENCE (idempotent MERGE per table)
# ==============================================================================

def ensure_gold_table(spark, table: str) -> str:
    path = gold_path(table)
    if not DeltaTable.isDeltaTable(spark, path):
        print(f"[Gold Persistence] Creating {table} at {path}")
        spark.createDataFrame([], GOLD_TABLES[table]["schema"]).write.format("delta").save(path)
    try:
        spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_table_name(table)} USING DELTA LOCATION '{path}'")
    except Exception:
        pass
    return path


def _metrics(target: "DeltaTable") -> Dict[str, int]:
    try:
        m = target.history(1).select("operationMetrics").first()["operationMetrics"]
        return {"inserted": int(m.get("numTargetRowsInserted", 0)), "updated": int(m.get("numTargetRowsUpdated", 0)),
                "deleted": int(m.get("numTargetRowsDeleted", 0))}
    except Exception as err:
        print(f"[Gold Persistence Warning] Could not read MERGE metrics: {err}")
        return {"inserted": 0, "updated": 0, "deleted": 0}


def merge_gold_table(spark, table: str, df_src: "DataFrame", window_start: Optional[str], window_end: Optional[str],
                     delete_missing: bool, country_filter: str = "ALL") -> Dict[str, int]:
    """
    MERGE on the table's business key (+ literal date window on calendar_date for fact/mart tables).
      WHEN MATCHED AND any non-key attribute changed (null-safe <=>) -> UPDATE   (batch id alone never triggers it)
      WHEN NOT MATCHED                                              -> INSERT
      WHEN NOT MATCHED BY SOURCE inside the window (fact / marts)   -> DELETE    (re-classified / removed rows)
    """
    spec = GOLD_TABLES[table]
    path = ensure_gold_table(spark, table)
    target = DeltaTable.forPath(spark, path)
    keys = spec["keys"]
    names = [f.name for f in spec["schema"].fields]
    audit_cols = {"load_timestamp", "_batch_id"}
    compare = [c for c in names if c not in keys and c not in audit_cols]

    windowed = spec["kind"] in ("fact", "mart") and window_start and window_end
    cond = " AND ".join(f"target.{k} = source.{k}" for k in keys)
    window_sql = (f"target.calendar_date >= '{window_start}' AND target.calendar_date <= '{window_end}'") if windowed else ""
    if windowed:
        cond += " AND " + window_sql

    changed = " OR ".join(f"NOT (target.{c} <=> source.{c})" for c in compare)
    update_set = {c: f"source.{c}" for c in compare + ["load_timestamp", "_batch_id"]}
    builder = (target.alias("target").merge(df_src.alias("source"), cond)
               .whenMatchedUpdate(condition=changed, set=update_set)
               .whenNotMatchedInsertAll())
    if delete_missing and spec["kind"] in ("fact", "mart"):
        # Scope the delete to the processed window AND country, so a country-filtered run never removes other countries' rows.
        delete_parts = ([window_sql] if windowed else []) + ([f"target.country_code = '{country_filter}'"] if country_filter != "ALL" else [])
        builder = builder.whenNotMatchedBySourceDelete(condition=" AND ".join(delete_parts) if delete_parts else None)
    builder.execute()
    m = _metrics(target)
    print(f"[Gold Persistence] {table}: {m}")
    return m


def table_file_stats(spark, table: str) -> Dict[str, Any]:
    try:
        row = spark.sql(f"DESCRIBE DETAIL delta.`{gold_path(table)}`").select("numFiles", "sizeInBytes").first()
        return {"numFiles": int(row["numFiles"]), "sizeInBytes": int(row["sizeInBytes"])}
    except Exception:
        return {"numFiles": None, "sizeInBytes": None}


def optimize_gold_tables(spark, tables: Optional[List[str]] = None) -> Dict[str, Dict[str, Any]]:
    """
    OPTIMIZE (+ ZORDER on the hot BI filter columns for fact/marts) and report file counts before/after.
    Never fails the run. Liquid clustering (CLUSTER BY) is the alternative on runtimes that support it.
    """
    report: Dict[str, Dict[str, Any]] = {}
    for table in (tables or list(GOLD_TABLES)):
        before = table_file_stats(spark, table)
        try:
            opt = DeltaTable.forPath(spark, gold_path(table)).optimize()
            z = GOLD_TABLES[table].get("zorder")
            opt.executeZOrderBy(*z) if z else opt.executeCompaction()
            status = "OPTIMIZED" + (f" ZORDER({', '.join(z)})" if z else "")
        except Exception as err:
            status = f"SKIPPED: {err}"
        report[table] = {"before": before, "after": table_file_stats(spark, table), "status": status}
        print(f"[Gold Optimize] {table}: {report[table]['status']} files {before['numFiles']} -> {report[table]['after']['numFiles']}")
    return report

In [0]:
# ==============================================================================
# STEP 6: INTEGRITY + RECONCILIATION GATES
# ==============================================================================

FK_CHECKS = [
    ("fact_network_anomaly", "date_time_key", "dim_date_time", "date_time_key"),
    ("fact_network_anomaly", "date_key", "dim_date", "date_key"),
    ("fact_network_anomaly", "country_code", "dim_country_region", "country_code"),
    ("fact_network_anomaly", "asn_id", "dim_isp_telecom", "asn_id"),
    ("fact_network_anomaly", "category_code", "dim_content_category", "category_code"),
    ("agg_daily_censorship_severity", "date_key", "dim_date", "date_key"),
    ("agg_daily_censorship_severity", "country_code", "dim_country_region", "country_code"),
    ("agg_daily_censorship_severity", "asn_id", "dim_isp_telecom", "asn_id"),
    ("agg_daily_censorship_severity", "category_code", "dim_content_category", "category_code"),
    ("agg_hourly_censorship_severity", "date_time_key", "dim_date_time", "date_time_key"),
    ("agg_hourly_censorship_severity", "country_code", "dim_country_region", "country_code"),
]


def count_orphans(df_child: "DataFrame", child_col: str, df_parent: "DataFrame", parent_col: str) -> int:
    return df_child.select(child_col).distinct().join(
        df_parent.select(F.col(parent_col).alias("_pk")).distinct(), F.col(child_col) == F.col("_pk"), "left_anti").count()


def validate_referential_integrity(spark, frames: Optional[Dict[str, "DataFrame"]] = None) -> Dict[str, int]:
    """Orphan count per foreign key. `frames` lets tests pass in-memory frames; default reads the Gold tables."""
    def get(name: str) -> "DataFrame":
        return frames[name] if frames else spark.read.format("delta").load(gold_path(name))
    return {f"{c}.{cc} -> {p}.{pc}": count_orphans(get(c), cc, get(p), pc) for c, cc, p, pc in FK_CHECKS}


def reconcile_counts(df_silver: "DataFrame", frames: Dict[str, "DataFrame"]) -> Dict[str, Any]:
    """Rows aggregated must tie out: marts sum to Silver; the fact equals the Silver anomaly count."""
    silver_total = df_silver.count()
    silver_anom = df_silver.where(F.col("is_anomaly")).count()
    daily = frames["agg_daily_censorship_severity"].agg(F.sum("total_measurements")).first()[0] or 0
    hourly = frames["agg_hourly_censorship_severity"].agg(F.sum("total_measurements")).first()[0] or 0
    fact = frames["fact_network_anomaly"].count()
    out = {"silver_rows": silver_total, "agg_daily_total": int(daily), "agg_hourly_total": int(hourly),
           "silver_anomalies": silver_anom, "fact_rows": fact}
    out["ok"] = (silver_total == out["agg_daily_total"] == out["agg_hourly_total"]) and silver_anom == fact
    return out

In [0]:
# ==============================================================================
# STEP 7: ORCHESTRATION + AUDIT LOGGING (layer = 'Silver-to-Gold')
# ==============================================================================

class _NullAuditor:
    def __enter__(self): return self
    def __exit__(self, *exc): return False
    def set_metrics(self, **kw): pass


def _make_auditor(session, parameter: str, load_type: str):
    if audit_logger and hasattr(audit_logger, "PipelineAuditor"):
        return audit_logger.PipelineAuditor(layer=GOLD_LAYER_NAME, parameter=parameter, load_type=load_type, spark=session)
    return _NullAuditor()


LOAD_ORDER = ["dim_date_time", "dim_date", "dim_country_region", "dim_isp_telecom", "dim_content_category",
              "fact_network_anomaly", "agg_daily_censorship_severity", "agg_hourly_censorship_severity"]


def run_silver_to_gold(
    spark: Optional[Any] = None, batch_date: Optional[str] = None, load_type: Optional[str] = None,
    lookback_days: Optional[int] = None, backfill_start: Optional[str] = None, backfill_end: Optional[str] = None,
    country_filter: Optional[str] = None, window_start: Optional[str] = None, window_end: Optional[str] = None,
    silver_path: Optional[str] = None, run_optimize: bool = True, fail_on_integrity_error: bool = True,
    allow_empty_purge: bool = False,
) -> Dict[str, Any]:
    """
    Refreshes the whole Gold star for the same date window Silver processed. Safe to re-run.
      standard   : run_silver_to_gold(batch_date="2026-09-28")                         # T-3 .. T
      from Silver: run_silver_to_gold(window_start=s["window_start"], window_end=s["window_end"])
      backfill   : run_silver_to_gold(load_type="backfill", backfill_start="2026-09-10", backfill_end="2026-09-14")
      rebuild    : run_silver_to_gold(load_type="full")
    """
    p = resolve_gold_parameters(batch_date, load_type, lookback_days, backfill_start, backfill_end,
                                country_filter, window_start, window_end)
    session = resolve_spark_session(spark)
    silver_path = silver_path or config.PATHS["silver"]
    load_ts = datetime.now(timezone.utc).replace(tzinfo=None)
    summary: Dict[str, Any] = dict(p)
    print(f"[Gold] batch_id={p['batch_id']} window=[{p['window_start']} .. {p['window_end']}] country={p['country_filter']}")

    df_silver = validate_silver_input(read_silver_window(session, silver_path, p["window_start"], p["window_end"], p["country_filter"]))
    silver_rows = df_silver.count()
    if silver_rows == 0 and not allow_empty_purge:
        print("[Gold] No Silver rows in the window: skipping refresh (existing Gold data is preserved).")
        summary.update({"status": "Skipped_NoSilverData", "silver_rows": 0, "tables": {}})
        with _make_auditor(session, p["batch_id"], p["load_type"]) as aud:
            aud.set_metrics(rows_read=0, rows_inserted=0, rows_updated=0, rows_quarantined=0)
        return summary

    frames = transform_silver_to_gold(df_silver, p["batch_id"], load_ts)
    recon = reconcile_counts(df_silver, frames)
    if not recon["ok"]:
        raise RuntimeError(f"[Gold] Reconciliation failed before write: {recon}")

    table_metrics: Dict[str, Dict[str, int]] = {}
    for table in LOAD_ORDER:
        df_t = frames[table]
        with _make_auditor(session, f"{p['batch_id']}|{table}", p["load_type"]) as aud:
            n_src = df_t.count()
            m = merge_gold_table(session, table, df_t, p["window_start"], p["window_end"], delete_missing=True,
                                 country_filter=p["country_filter"])
            # rows_read = Silver rows aggregated for this table; execution time = audit start/end.
            # The audit schema has no "deleted" column, so rows removed by NOT MATCHED BY SOURCE are counted in rows_updated.
            aud.set_metrics(rows_read=(silver_rows if GOLD_TABLES[table]["kind"] != "dim" else n_src),
                            rows_inserted=m["inserted"], rows_updated=m["updated"] + m["deleted"], rows_quarantined=0)
            table_metrics[table] = {**m, "source_rows": n_src}

    orphans = validate_referential_integrity(session)
    bad = {k: v for k, v in orphans.items() if v}
    if bad and fail_on_integrity_error:
        raise RuntimeError(f"[Gold] Referential integrity violated (orphan keys): {bad}")
    opt = optimize_gold_tables(session) if run_optimize else {}

    summary.update({"status": "Success", "silver_rows": silver_rows, "tables": table_metrics,
                    "reconciliation": recon, "orphans": orphans, "optimize": opt})
    return summary

In [0]:
if __name__ == "__main__":
    print("=" * 75)
    print(" PROJECT PANOPTICON: GOLD DIMENSIONAL MODELING (Silver -> Gold)")
    print("=" * 75)
    init_gold_widgets()
    result = run_silver_to_gold()
    for k, v in result.items():
        print(f"  * {k:<16}: {v}")